In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample

In [2]:
# --- Load data ---
db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'
engine = create_engine(db_path)

sql_query = """
SELECT *
FROM geographic_features
LEFT JOIN weather_features USING (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
"""

with engine.connect() as connection:
    MD_df = pd.read_sql_query(text(sql_query), connection)

# --- Standard cleaning pipeline from earlier in the programme ---
MD_df.rename(columns={'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, inplace=True)
MD_df.rename(columns={'Crop_type_Temp': 'Crop_type'}, inplace=True)
MD_df['Elevation'] = MD_df['Elevation'].abs()

def correct_crop_type(crop):
    corrections = {'cassaval': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
    return corrections.get(str(crop).strip(), str(crop).strip())

MD_df['Crop_type'] = MD_df['Crop_type'].apply(correct_crop_type)

print(f"DataFrame shape: {MD_df.shape}")
print(f"\nCrop type distribution:")
print(MD_df['Crop_type'].value_counts())

DataFrame shape: (5654, 18)

Crop type distribution:
Crop_type
wheat      1342
tea         913
potato      823
cassava     672
banana      633
coffee      607
maize       399
rice        265
Name: count, dtype: int64


In [3]:
df = MD_df 
display(df.head())

,Field_ID,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield
0,40734,786.05580,-7.389911,-7.556202,Rural_Akatsi,14.795113,1125.2,-3.1,33.1,15.00,0.62,Sandy,6.169393,0.085267,1.3,0.751354,cassava,0.577964
1,30629,674.33410,-7.736849,-1.051539,Rural_Sokoto,11.374611,1450.7,-3.9,30.6,13.35,0.64,Volcanic,5.676648,0.399684,2.2,1.069865,cassava,0.486302
2,39924,826.53390,-9.926616,0.115156,Rural_Sokoto,11.339692,2208.9,-1.8,28.4,13.30,0.69,Volcanic,5.331993,0.358029,3.4,2.208801,tea,0.649647
3,5754,574.94617,-2.420131,-6.592215,Rural_Kilimani,7.109855,328.8,-5.8,32.2,13.20,0.54,Loamy,5.328150,0.286687,2.4,1.277635,cassava,0.532348
4,14146,886.35300,-3.055434,-7.952609,Rural_Kilimani,55.007656,785.2,-2.5,31.0,14.25,0.72,Sandy,5.721234,0.043190,1.5,0.832614,wheat,0.555076


In [5]:
# Check the dtatypes of the columns
for col in df.columns:
    print(f"{col:<20}: {df[col].dtype}")

Field_ID            : int64
Elevation           : float64
Latitude            : float64
Longitude           : float64
Location            : str
Slope               : float64
Rainfall            : float64
Min_temperature_C   : float64
Max_temperature_C   : float64
Ave_temps           : float64
Soil_fertility      : float64
Soil_type           : str
pH                  : float64
Pollution_level     : float64
Plot_size           : float64
Annual_yield        : float64
Crop_type           : str
Standard_yield      : float64


# **[Analysis]()**

## **Challenge 1: Feature Selection and Encoding**

Before modeling, we choose our features and prepare the target. We use the environmental and geographic features Sanaa has curated over earlier courses, and encode the crop target as numeric labels.

### **Task**
Create a function `prepare_crop_features` that:
- Takes the full Maji Ndogo DataFrame.
- Selects these feature columns: `Elevation`, `Rainfall`, `Min_temperature_C`, `Max_temperature_C`, `Ave_temps`, `Soil_fertility`, `pH`, `Pollution_level`, `Slope`, `Plot_size`.
- Drops rows with any missing values in the selected columns or the target `Crop_type`.
- Encodes `Crop_type` using a `LabelEncoder` into a new column `Crop_encoded`.
- Returns `(df_prepared, feature_cols, le)`, where `df_prepared` contains the feature columns **and** `Crop_encoded`, `feature_cols` is the feature-name list, and `le` is the fitted `LabelEncoder`.
    

In [6]:
def prepare_crop_features(df):
    feature_cols = [
        "Elevation",
        "Rainfall",
        "Min_temperature_C",
        "Max_temperature_C",
        "Ave_temps",
        "Soil_fertility",
        "pH",
        "Pollution_level",
        "Slope",
        "Plot_size"
    ]

    # Select features and target, then remove missing values
    df_prepared = df[feature_cols + ["Crop_type"]].dropna().copy()

    # Encode Crop_type
    le = LabelEncoder()
    df_prepared["Crop_encoded"] = le.fit_transform(
        df_prepared["Crop_type"]
    )

    # Keep only features and encoded target
    df_prepared = df_prepared[
        feature_cols + ["Crop_encoded"]
    ]

    return df_prepared, feature_cols, le

In [7]:
df_prepared, feature_cols, le = prepare_crop_features(df)

print("Prepared DataFrame shape:", df_prepared.shape)
print("Feature columns:", feature_cols)
print("Crop classes:", le.classes_)

print("\nEncoded target distribution:")
print(
    df_prepared["Crop_encoded"]
    .map(dict(enumerate(le.classes_)))
    .value_counts()
)

Prepared DataFrame shape: (5654, 11)
Feature columns: ['Elevation', 'Rainfall', 'Min_temperature_C', 'Max_temperature_C', 'Ave_temps', 'Soil_fertility', 'pH', 'Pollution_level', 'Slope', 'Plot_size']
Crop classes: ['banana' 'cassava' 'coffee' 'maize' 'potato' 'rice' 'tea' 'wheat']

Encoded target distribution:
Crop_encoded
wheat      1342
tea         913
potato      823
cassava     672
banana      633
coffee      607
maize       399
rice        265
Name: count, dtype: int64


 ## Challenge 2: Training a Multiclass Logistic Regression Classifier

Logistic Regression handles multiclass problems by fitting one classifier per class (one-vs-rest) or by extending to a softmax across all classes at once; `sklearn` manages this automatically.

### Task
Create a function `train_crop_classifier` that:
- Takes `df_prepared`, `feature_cols`, `random_state` (default `42`), and `model_type` (`'logistic'` or `'random_forest'`).
- Splits the data 80–20 (features = `feature_cols`, target = `Crop_encoded`), using `random_state`.
- Trains the selected model: `LogisticRegression(max_iter=1000)` or `RandomForestClassifier(n_estimators=100, random_state=random_state)`.
- Returns `(model, X_test, y_test)`.